# 9. Publish to DeltaBay

Final step of the **altendor** pipeline. This notebook:

1. Picks the latest run that produced a `debate.json` (from notebook 8).
2. Copies it into the DeltaBay frontend at `frontend/app/src/data/altendor/debate.json`.
3. Runs the `funding-debate.altendor` vitest to confirm the loader is happy with the new file.
4. Prints the dev-server command so you can inspect the populated debate in the UI.

**Prerequisites**

- Notebook 8 must have produced `output/<RUN_ID>/debate.json`.
- `pnpm` and `node` available on `PATH` (provided by the Nix devshell).

## Setup

Pin paths, pick the latest run that actually has a `debate.json`, and bail loudly if none exists.

In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd().parent.resolve()
OUTPUT_ROOT = REPO_ROOT / "altendor" / "output"
DELTABAY_ROOT = Path("/home/automathis/Documents/Codebases/DeltaBay")

candidates = sorted(
    (p.parent for p in OUTPUT_ROOT.glob("*/debate.json")),
    key=lambda p: p.name,
)
if not candidates:
    raise FileNotFoundError(
        f"No run under {OUTPUT_ROOT} contains debate.json. Run notebook 8 first."
    )

OUT_DIR = candidates[-1]
RUN_ID = OUT_DIR.name
print(f"Run id: {RUN_ID}\nOutput dir: {OUT_DIR}")

## Copy `debate.json` into the DeltaBay frontend

Mirrors the latest run's `debate.json` to `DeltaBay/frontend/app/src/data/altendor/debate.json`. The destination directory is created if missing.

In [ ]:
import shutil
from pathlib import Path

DELTABAY_ROOT = Path("/home/automathis/Documents/Codebases/DeltaBay")
src = OUT_DIR / "debate.json"
dst = DELTABAY_ROOT / "frontend" / "app" / "src" / "data" / "altendor" / "debate.json"
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(src, dst)
print(f"Copied:\n  {src}\n  \u2192 {dst}")

## Vitest sanity check

Runs the `funding-debate.altendor` vitest in `DeltaBay/frontend/app` so we catch any shape mismatch before opening the UI. Raises if the test suite fails.

In [ ]:
import subprocess

result = subprocess.run(
    ["pnpm", "-C", str(DELTABAY_ROOT / "frontend" / "app"),
     "vitest", "run", "funding-debate.altendor"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:")
    print(result.stderr)
    raise RuntimeError(f"vitest failed (exit {result.returncode})")
print("\u2713 vitest passed")

## Inspect the populated debate

The DeltaBay dev server isn't launched from the notebook — keep that to your own terminal so you control its lifecycle. The next cell just prints the command.

In [ ]:
print("Launch the DeltaBay dev server in a terminal:\n")
print(f"  pnpm -C {DELTABAY_ROOT / 'frontend' / 'app'} dev")
print("\nThen open the 'optimal scientific system' debate in the UI.")